In [22]:
# Cell 1 - setup

import io
import json
import random
import contextlib
import ollama

MODEL = "gemma4:31b-cloud"  #     gemma4:31b-cloud    gpt-oss:120b-cloud


In [2]:
# Cell 2 - our 'database': 500 customers, each with a spending total

random.seed(1)

FIRST = ["Rahul", "Priya", "Anand", "Sneha", "Vikram", "Divya", "Karan",
         "Meera", "Arjun", "Anita"]
LAST = ["Mehta", "Shah", "Kumar", "Nair", "Reddy", "Iyer", "Bose", "Rao"]

CUSTOMERS = []
SPENDING = {}
for i in range(500):
    cid = f"C-{1000 + i}"
    CUSTOMERS.append({"id": cid,
                       "name": f"{random.choice(FIRST)} {random.choice(LAST)}"})
    SPENDING[cid] = random.randint(5000, 250000)   # rupees spent

# how many are ACTUALLY above 1 lakh? (so we can check the answer later)
true_answer = [c["name"] for c in CUSTOMERS if SPENDING[c["id"]] > 100000]
print(f"500 customers loaded. {len(true_answer)} of them spent over 1 lakh.")

500 customers loaded. 316 of them spent over 1 lakh.


In [3]:
SPENDING

{'C-1000': 71864,
 'C-1001': 204481,
 'C-1002': 175811,
 'C-1003': 29604,
 'C-1004': 239177,
 'C-1005': 164236,
 'C-1006': 74816,
 'C-1007': 241303,
 'C-1008': 10851,
 'C-1009': 236010,
 'C-1010': 115655,
 'C-1011': 205199,
 'C-1012': 149928,
 'C-1013': 65520,
 'C-1014': 80964,
 'C-1015': 224566,
 'C-1016': 53734,
 'C-1017': 199811,
 'C-1018': 138095,
 'C-1019': 79490,
 'C-1020': 226825,
 'C-1021': 159403,
 'C-1022': 68632,
 'C-1023': 179258,
 'C-1024': 148864,
 'C-1025': 120071,
 'C-1026': 209064,
 'C-1027': 102130,
 'C-1028': 128029,
 'C-1029': 189387,
 'C-1030': 174648,
 'C-1031': 136658,
 'C-1032': 206988,
 'C-1033': 111025,
 'C-1034': 227156,
 'C-1035': 125358,
 'C-1036': 105581,
 'C-1037': 140968,
 'C-1038': 116697,
 'C-1039': 233067,
 'C-1040': 137309,
 'C-1041': 218215,
 'C-1042': 95722,
 'C-1043': 125100,
 'C-1044': 215913,
 'C-1045': 149377,
 'C-1046': 230708,
 'C-1047': 13508,
 'C-1048': 232561,
 'C-1049': 8816,
 'C-1050': 75422,
 'C-1051': 95288,
 'C-1052': 48901,
 'C-1053'

In [4]:
# Cell 3 - the two underlying tools (plain Python functions)

def get_all_customers():
    """Get the list of all customers (id and name)."""
    return CUSTOMERS


def get_total_spent(customer_id: str):
    """Get the total amount (in rupees) one customer has spent."""
    return SPENDING.get(customer_id, 0)

In [5]:
NORMAL_TOOLS = [get_all_customers, get_total_spent]
NORMAL_REGISTRY = {f.__name__: f for f in NORMAL_TOOLS}

NORMAL_SYSTEM = (
    "You are a data assistant. Answer the user's question using the tools. "
    "get_all_customers lists customers; get_total_spent gives one customer's "
    "spending. Work through customers to find who qualifies."
)

In [6]:
NORMAL_REGISTRY

{'get_all_customers': <function __main__.get_all_customers()>,
 'get_total_spent': <function __main__.get_total_spent(customer_id: str)>}

In [7]:
import json

messages = [
    {"role": "system", "content": NORMAL_SYSTEM},
    {"role": "user", "content": "Which customers have spent more than 100000 rupees? List their names."}
]

response = ollama.chat(model=MODEL, messages=messages, tools=NORMAL_TOOLS)
messages.append(response.message)
print(response)
print("*********"*10)
print("tokens:", response.prompt_eval_count)



model='gemma4:31b' created_at='2026-08-04T05:50:32.436061975Z' done=True done_reason='stop' total_duration=12887629047 load_duration=None prompt_eval_count=171 prompt_eval_duration=None eval_count=11 eval_duration=None message=Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='get_all_customers', arguments={}))])
******************************************************************************************
tokens: 171


In [8]:
for call in response.message.tool_calls:
    name = call.function.name
    args = call.function.arguments
    result = NORMAL_REGISTRY[name](**args)
    messages.append({"role": "tool", "tool_name": name, "content": json.dumps(result)})
    print(result)

[{'id': 'C-1000', 'name': 'Anand Shah'}, {'id': 'C-1001', 'name': 'Priya Rao'}, {'id': 'C-1002', 'name': 'Meera Rao'}, {'id': 'C-1003', 'name': 'Karan Nair'}, {'id': 'C-1004', 'name': 'Meera Mehta'}, {'id': 'C-1005', 'name': 'Karan Bose'}, {'id': 'C-1006', 'name': 'Rahul Rao'}, {'id': 'C-1007', 'name': 'Sneha Shah'}, {'id': 'C-1008', 'name': 'Divya Mehta'}, {'id': 'C-1009', 'name': 'Rahul Mehta'}, {'id': 'C-1010', 'name': 'Karan Nair'}, {'id': 'C-1011', 'name': 'Rahul Nair'}, {'id': 'C-1012', 'name': 'Meera Rao'}, {'id': 'C-1013', 'name': 'Sneha Iyer'}, {'id': 'C-1014', 'name': 'Sneha Rao'}, {'id': 'C-1015', 'name': 'Rahul Bose'}, {'id': 'C-1016', 'name': 'Arjun Shah'}, {'id': 'C-1017', 'name': 'Vikram Shah'}, {'id': 'C-1018', 'name': 'Divya Bose'}, {'id': 'C-1019', 'name': 'Sneha Reddy'}, {'id': 'C-1020', 'name': 'Anita Rao'}, {'id': 'C-1021', 'name': 'Arjun Bose'}, {'id': 'C-1022', 'name': 'Rahul Rao'}, {'id': 'C-1023', 'name': 'Karan Bose'}, {'id': 'C-1024', 'name': 'Anand Iyer'}, {

In [9]:
print(messages)

[{'role': 'system', 'content': "You are a data assistant. Answer the user's question using the tools. get_all_customers lists customers; get_total_spent gives one customer's spending. Work through customers to find who qualifies."}, {'role': 'user', 'content': 'Which customers have spent more than 100000 rupees? List their names.'}, Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='get_all_customers', arguments={}))]), {'role': 'tool', 'tool_name': 'get_all_customers', 'content': '[{"id": "C-1000", "name": "Anand Shah"}, {"id": "C-1001", "name": "Priya Rao"}, {"id": "C-1002", "name": "Meera Rao"}, {"id": "C-1003", "name": "Karan Nair"}, {"id": "C-1004", "name": "Meera Mehta"}, {"id": "C-1005", "name": "Karan Bose"}, {"id": "C-1006", "name": "Rahul Rao"}, {"id": "C-1007", "name": "Sneha Shah"}, {"id": "C-1008", "name": "Divya Mehta"}, {"id": "C-1009", "name": "Rahul Mehta"}, {"id": "C-1010", "name": "Karan Nair

In [10]:
response = ollama.chat(model=MODEL, messages=messages, tools=NORMAL_TOOLS)


messages.append(response.message)
print(response)
print("******"*10)
print("tokens:", response.prompt_eval_count)

model='gemma4:31b' created_at='2026-08-04T05:52:22.90413377Z' done=True done_reason='stop' total_duration=110022198406 load_duration=None prompt_eval_count=9594 prompt_eval_duration=None eval_count=11505 eval_duration=None message=Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='get_total_spent', arguments={'customer_id': 'C-1000'})), ToolCall(function=Function(name='get_total_spent', arguments={'customer_id': 'C-1001'})), ToolCall(function=Function(name='get_total_spent', arguments={'customer_id': 'C-1002'})), ToolCall(function=Function(name='get_total_spent', arguments={'customer_id': 'C-1003'})), ToolCall(function=Function(name='get_total_spent', arguments={'customer_id': 'C-1004'})), ToolCall(function=Function(name='get_total_spent', arguments={'customer_id': 'C-1005'})), ToolCall(function=Function(name='get_total_spent', arguments={'customer_id': 'C-1006'})), ToolCall(function=Function(name='get_total_sp

In [12]:
print(response.message.tool_calls)

[ToolCall(function=Function(name='get_total_spent', arguments={'customer_id': 'C-1000'})), ToolCall(function=Function(name='get_total_spent', arguments={'customer_id': 'C-1001'})), ToolCall(function=Function(name='get_total_spent', arguments={'customer_id': 'C-1002'})), ToolCall(function=Function(name='get_total_spent', arguments={'customer_id': 'C-1003'})), ToolCall(function=Function(name='get_total_spent', arguments={'customer_id': 'C-1004'})), ToolCall(function=Function(name='get_total_spent', arguments={'customer_id': 'C-1005'})), ToolCall(function=Function(name='get_total_spent', arguments={'customer_id': 'C-1006'})), ToolCall(function=Function(name='get_total_spent', arguments={'customer_id': 'C-1007'})), ToolCall(function=Function(name='get_total_spent', arguments={'customer_id': 'C-1008'})), ToolCall(function=Function(name='get_total_spent', arguments={'customer_id': 'C-1009'})), ToolCall(function=Function(name='get_total_spent', arguments={'customer_id': 'C-1010'})), ToolCall(

In [13]:
messages

[{'role': 'system',
  'content': "You are a data assistant. Answer the user's question using the tools. get_all_customers lists customers; get_total_spent gives one customer's spending. Work through customers to find who qualifies."},
 {'role': 'user',
  'content': 'Which customers have spent more than 100000 rupees? List their names.'},
 Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='get_all_customers', arguments={}))]),
 {'role': 'tool',
  'tool_name': 'get_all_customers',
  'content': '[{"id": "C-1000", "name": "Anand Shah"}, {"id": "C-1001", "name": "Priya Rao"}, {"id": "C-1002", "name": "Meera Rao"}, {"id": "C-1003", "name": "Karan Nair"}, {"id": "C-1004", "name": "Meera Mehta"}, {"id": "C-1005", "name": "Karan Bose"}, {"id": "C-1006", "name": "Rahul Rao"}, {"id": "C-1007", "name": "Sneha Shah"}, {"id": "C-1008", "name": "Divya Mehta"}, {"id": "C-1009", "name": "Rahul Mehta"}, {"id": "C-1010", "name": 

In [14]:
for call in response.message.tool_calls:
    name = call.function.name
    args = call.function.arguments
    result = NORMAL_REGISTRY[name](**args)
    messages.append({"role": "tool", "tool_name": name, "content": json.dumps(result)})
    print(result)


71864
204481
175811
29604
239177
164236
74816
241303
10851
236010
115655
205199
149928
65520
80964
224566
53734
199811
138095
79490
226825
159403
68632
179258
148864
120071
209064
102130
128029
189387
174648
136658
206988
111025
227156
125358
105581
140968
116697
233067
137309
218215
95722
125100
215913
149377
230708
13508
232561
8816
75422
95288
48901
143249
174922
189189
129196
86790
115341
33511
163767
64080
43394
121829
147790
63509
108520
177968
198318
60609
23540
245432
114097
39181
159818
49962
104082
30958
160034
32374
137148
90287
240845
46147
217635
93891
74870
249367
244716
206313
22123
39869
243809
75257
101497
34861
232388
40480
89076
24187
222187
35065
25093
153364
100654
125001
211311
8245
29034
221562
67818
47472
48878
200036
243751
149234
87433
175931
12146
195443
122924
109478
244440
34192
210818
178495
53030
85562
99493
28439
175920
64618
15761
88030
69447
147665
69250
216881
23960
196147
171561
201799
127902
136446
138502
208478
220364
33017
38108
147997
220228
5170

In [15]:
print(messages)

[{'role': 'system', 'content': "You are a data assistant. Answer the user's question using the tools. get_all_customers lists customers; get_total_spent gives one customer's spending. Work through customers to find who qualifies."}, {'role': 'user', 'content': 'Which customers have spent more than 100000 rupees? List their names.'}, Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='get_all_customers', arguments={}))]), {'role': 'tool', 'tool_name': 'get_all_customers', 'content': '[{"id": "C-1000", "name": "Anand Shah"}, {"id": "C-1001", "name": "Priya Rao"}, {"id": "C-1002", "name": "Meera Rao"}, {"id": "C-1003", "name": "Karan Nair"}, {"id": "C-1004", "name": "Meera Mehta"}, {"id": "C-1005", "name": "Karan Bose"}, {"id": "C-1006", "name": "Rahul Rao"}, {"id": "C-1007", "name": "Sneha Shah"}, {"id": "C-1008", "name": "Divya Mehta"}, {"id": "C-1009", "name": "Rahul Mehta"}, {"id": "C-1010", "name": "Karan Nair

In [16]:
response = ollama.chat(model=MODEL, messages=messages, tools=NORMAL_TOOLS)
print(response)

model='gemma4:31b' created_at='2026-08-04T05:55:33.85125874Z' done=True done_reason='stop' total_duration=4285737433 load_duration=None prompt_eval_count=31402 prompt_eval_duration=None eval_count=1 eval_duration=None message=Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=None)


# Way 1 - Normal Agentic FLow 

In [20]:
import json

def run_normal(question, max_turns=20):
    messages = [
        {"role": "system", "content": NORMAL_SYSTEM},
        {"role": "user", "content": question}
    ]
    round_trips = 0

    for turn in range(1, max_turns + 1):
        response = ollama.chat(model=MODEL, messages=messages, tools=NORMAL_TOOLS)
        size = response.prompt_eval_count or 0
        n_calls = len(response.message.tool_calls or [])
        print(f"turn {turn}: {size:>6} tokens | {n_calls} tool call(s) this turn")
        messages.append(response.message)

        if not response.message.tool_calls:
            answer = response.message.content
            print(f"\nANSWER: {answer}\n")
            break

        round_trips += n_calls          # FIX 1: count calls, not turns

        for call in response.message.tool_calls:
            name = call.function.name
            args = call.function.arguments
            result = NORMAL_REGISTRY[name](**args)
            messages.append({
                "role": "tool",
                "tool_name": name,
                "content": json.dumps(result)
            })
    else:
        print(f"\n(stopped at max_turns ({max_turns}) - it needs more calls to finish)\n")

    print(f"Total tool round trips: {round_trips}")

In [21]:
import time

start= time.time()
run_normal("Which customers have spent more than 100000 rupees? List their names.")
end= time.time()
difference= end-start

print(f"Time_Taken:- {difference}")

turn 1:    210 tokens | 1 tool call(s) this turn
turn 2:   9117 tokens | 1 tool call(s) this turn
turn 3:   9396 tokens | 1 tool call(s) this turn
turn 4:   9450 tokens | 1 tool call(s) this turn
turn 5:   9527 tokens | 1 tool call(s) this turn
turn 6:   9592 tokens | 1 tool call(s) this turn
turn 7:   9640 tokens | 1 tool call(s) this turn
turn 8:   9698 tokens | 1 tool call(s) this turn
turn 9:   9748 tokens | 1 tool call(s) this turn
turn 10:   9796 tokens | 1 tool call(s) this turn
turn 11:   9846 tokens | 1 tool call(s) this turn
turn 12:   9894 tokens | 1 tool call(s) this turn
turn 13:   9944 tokens | 1 tool call(s) this turn
turn 14:  10008 tokens | 1 tool call(s) this turn
turn 15:  10058 tokens | 1 tool call(s) this turn
turn 16:  10115 tokens | 1 tool call(s) this turn
turn 17:  10357 tokens | 1 tool call(s) this turn
turn 18:  10405 tokens | 1 tool call(s) this turn
turn 19:  10454 tokens | 1 tool call(s) this turn
turn 20:  10501 tokens | 1 tool call(s) this turn

(stopped

# Way 2 - Programmatic tool calling

In [24]:
# Cell 5 - the run_code tool (this IS the mechanism)

def run_code(python_code: str):  # takes python code as input [its the code that AI generated]
    """Run Python code to answer questions that need computation over data.

    These functions are already available inside your code:
      get_all_customers()        -> list of {id, name}
      get_total_spent(customer_id) -> int, rupees spent

    Write code that computes the answer and PRINTS it. Only what you print
    is returned to you - the raw data stays inside the code.

    Args:
        python_code: The Python code to execute.

    Returns:
        A dict with the printed output, or an error message.
    """

    #CREATING AN SANDBOX NAMED WORKSPACE AND RATHER THAN GIVING ALL THE TOOLS TO LLM , WE MENTION THE TOOLS IN THE SANDBOX
    
    workspace = {
        "get_all_customers": get_all_customers,
        "get_total_spent": get_total_spent,
    }

    buffer = io.StringIO()   #CREATE AN TEMPORARY BUFFER MEMORY IN THE RAM, IT IS TEMPORARY , FAST AND DISAPPEAR WHEN THE FUNCTION IS DONE.
    try:
        with contextlib.redirect_stdout(buffer):   # for the next few lines , whatever the code try to print , dont show it in line but sent it to buffer
            exec(python_code, workspace)            # Takes the python code(send by user and run it in workspace)
        out = buffer.getvalue().strip() #after code have run , get the code from the buffer memory adn store it into variable called out
        return {"result": out or "(code ran but printed nothing)"}  # return variable out.
    except Exception as e:
        # errors are context too - hand them back so the model can fix its code
        return {"error": f"{type(e).__name__}: {e}"}

In [25]:
PROG_TOOLS = [run_code]

PROG_SYSTEM = (
    "You are a data assistant. You have ONE tool: run_code. To answer "
    "questions about customer data, write Python that uses the available "
    "functions get_all_customers() and get_total_spent(id), and print the "
    "final answer. Do not try to list raw data yourself - compute it in code."
)


In [41]:
# Start the conversation: system prompt + the user's question.

#question = "Which customers have spent more than 100000 rupees? List their names."
#question = "Which customers have spent more than 100000 rupees? List their names and give their count too"
question = "Which customers have spent more than 100000 rupees? List their names along with their spend and give total count too."

messages = [{"role": "system", "content": PROG_SYSTEM},
            {"role": "user", "content": question}]


In [42]:
# Send it to the model. It should decide to WRITE CODE (a run_code tool call).


MODEL= "gemma4:31b-cloud"

response = ollama.chat(model=MODEL, messages=messages, tools=PROG_TOOLS)
messages.append(response.message)

print("tokens in context:", response.prompt_eval_count)
print("did it call a tool?:", bool(response.message.tool_calls))

print(response)

tokens in context: 246
did it call a tool?: True
model='gemma4:31b' created_at='2026-08-04T06:28:00.909717189Z' done=True done_reason='stop' total_duration=1341745925 load_duration=None prompt_eval_count=246 prompt_eval_duration=None eval_count=128 eval_duration=None message=Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='run_code', arguments={'python_code': 'customers = get_all_customers()\nhigh_spenders = []\n\nfor customer in customers:\n    spent = get_total_spent(customer[\'id\'])\n    if spent > 100000:\n        high_spenders.append((customer[\'name\'], spent))\n\nprint(f"High Spenders:")\nfor name, spent in high_spenders:\n    print(f"{name}: {spent} rupees")\nprint(f"Total Count: {len(high_spenders)}")'}))])


In [43]:
# Pull out the Python the model wrote, and just LOOK at it before running.

call = response.message.tool_calls[0]
code = call.function.arguments["python_code"]

print("--- the model wrote this code: ---\n")
print(code)

--- the model wrote this code: ---

customers = get_all_customers()
high_spenders = []

for customer in customers:
    spent = get_total_spent(customer['id'])
    if spent > 100000:
        high_spenders.append((customer['name'], spent))

print(f"High Spenders:")
for name, spent in high_spenders:
    print(f"{name}: {spent} rupees")
print(f"Total Count: {len(high_spenders)}")


In [44]:
# Now actually run it. The 500 tool calls happen INSIDE here.
# Only what the code prints comes back.

result = run_code(code)

print("--- what the sandbox returned: ---")
print(result)

--- what the sandbox returned: ---
{'result': 'High Spenders:\nPriya Rao: 204481 rupees\nMeera Rao: 175811 rupees\nMeera Mehta: 239177 rupees\nKaran Bose: 164236 rupees\nSneha Shah: 241303 rupees\nRahul Mehta: 236010 rupees\nKaran Nair: 115655 rupees\nRahul Nair: 205199 rupees\nMeera Rao: 149928 rupees\nRahul Bose: 224566 rupees\nVikram Shah: 199811 rupees\nDivya Bose: 138095 rupees\nAnita Rao: 226825 rupees\nArjun Bose: 159403 rupees\nKaran Bose: 179258 rupees\nAnand Iyer: 148864 rupees\nDivya Shah: 120071 rupees\nArjun Shah: 209064 rupees\nAnand Bose: 102130 rupees\nMeera Mehta: 128029 rupees\nRahul Reddy: 189387 rupees\nAnita Bose: 174648 rupees\nAnand Kumar: 136658 rupees\nSneha Mehta: 206988 rupees\nSneha Nair: 111025 rupees\nArjun Iyer: 227156 rupees\nAnita Iyer: 125358 rupees\nVikram Mehta: 105581 rupees\nArjun Kumar: 140968 rupees\nArjun Nair: 116697 rupees\nRahul Rao: 233067 rupees\nDivya Nair: 137309 rupees\nKaran Rao: 218215 rupees\nRahul Iyer: 125100 rupees\nAnita Mehta: 21

In [45]:
# Feed the printed answer back into the conversation, so the model can read it.

messages.append({"role": "tool", "tool_name": "run_code",
                 "content": json.dumps(result)})

print("messages so far:", len(messages))

messages so far: 4


In [46]:
messages

[{'role': 'system',
  'content': 'You are a data assistant. You have ONE tool: run_code. To answer questions about customer data, write Python that uses the available functions get_all_customers() and get_total_spent(id), and print the final answer. Do not try to list raw data yourself - compute it in code.'},
 {'role': 'user',
  'content': 'Which customers have spent more than 100000 rupees? List their names along with their spend and give total count too.'},
 Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='run_code', arguments={'python_code': 'customers = get_all_customers()\nhigh_spenders = []\n\nfor customer in customers:\n    spent = get_total_spent(customer[\'id\'])\n    if spent > 100000:\n        high_spenders.append((customer[\'name\'], spent))\n\nprint(f"High Spenders:")\nfor name, spent in high_spenders:\n    print(f"{name}: {spent} rupees")\nprint(f"Total Count: {len(high_spenders)}")'}))]),
 {'r

In [47]:
# The model reads the code's result and writes the final answer for the user.

response = ollama.chat(model=MODEL, messages=messages, tools=PROG_TOOLS)

print("tokens in context:", response.prompt_eval_count)
print("\nANSWER:\n", response.message.content)

tokens in context: 4724

ANSWER:
 The customers who have spent more than 100,000 rupees are as follows:

*   **Priya Rao**: 204,481 rupees
*   **Meera Rao**: 175,811 rupees
*   **Meera Mehta**: 239,177 rupees
*   **Karan Bose**: 164,236 rupees
*   **Sneha Shah**: 241,303 rupees
*   **Rahul Mehta**: 236,010 rupees
*   **Karan Nair**: 115,655 rupees
*   **Rahul Nair**: 205,199 rupees
*   **Meera Rao**: 149,928 rupees
*   **Rahul Bose**: 224,566 rupees
*   **Vikram Shah**: 199,811 rupees
*   **Divya Bose**: 138,095 rupees
*   **Anita Rao**: 226,825 rupees
*   **Arjun Bose**: 159,403 rupees
*   **Karan Bose**: 179,258 rupees
*   **Anand Iyer**: 148,864 rupees
*   **Divya Shah**: 120,071 rupees
*   **Arjun Shah**: 209,064 rupees
*   **Anand Bose**: 102,130 rupees
*   **Meera Mehta**: 128,029 rupees
*   **Rahul Reddy**: 189,387 rupees
*   **Anita Bose**: 174,648 rupees
*   **Anand Kumar**: 136,658 rupees
*   **Sneha Mehta**: 206,988 rupees
*   **Sneha Nair**: 111,025 rupees
*   **Arjun Iyer*

In [48]:
# Cell 6 - programmatic tool calling

def run_programmatic(question, max_turns=5):
    messages = [{"role": "system", "content": PROG_SYSTEM},
                {"role": "user", "content": question}]
    round_trips = 0

    for turn in range(1, max_turns + 1):
        response = ollama.chat(model=MODEL, messages=messages, tools=PROG_TOOLS)
        round_trips += 1
        size = response.prompt_eval_count or 0
        print(f"turn {turn}: {size:>6} tokens in context")

        messages.append(response.message)
        if not response.message.tool_calls:
            print(f"\nANSWER: {response.message.content}\n")
            break

        for call in response.message.tool_calls:
            if call.function.name == "run_code":
                code = call.function.arguments["python_code"]
                print("\n--- the model wrote this code: ---")
                print(code)
                print("--- running it in the sandbox ---\n")
                result = run_code(code)
                messages.append({"role": "tool", "tool_name": "run_code",
                                  "content": json.dumps(result)})

    print(f"total round trips to the model: {round_trips}")


In [49]:
run_programmatic("Which customers have spent more than 100000 rupees? List their names.")
# run_programmatic("Which customers have spent more than 100000 rupees? List their names and give their count too")
# run_programmatic("Which customers have spent more than 100000 rupees? List their names along with their spend and give total count too.")

turn 1:    237 tokens in context

--- the model wrote this code: ---
customers = get_all_customers()
high_spenders = []

for customer in customers:
    spent = get_total_spent(customer['id'])
    if spent > 100000:
        high_spenders.append(customer['name'])

print(high_spenders)
--- running it in the sandbox ---

turn 2:   1850 tokens in context

ANSWER: The customers who have spent more than 100,000 rupees are:

Priya Rao, Meera Rao, Meera Mehta, Karan Bose, Sneha Shah, Rahul Mehta, Karan Nair, Rahul Nair, Meera Rao, Rahul Bose, Vikram Shah, Divya Bose, Anita Rao, Arjun Bose, Karan Bose, Anand Iyer, Divya Shah, Arjun Shah, Anand Bose, Meera Mehta, Rahul Reddy, Anita Bose, Anand Kumar, Sneha Mehta, Sneha Nair, Arjun Iyer, Anita Iyer, Vikram Mehta, Arjun Kumar, Arjun Nair, Rahul Rao, Divya Nair, Karan Rao, Rahul Iyer, Anita Mehta, Sneha Kumar, Anita Kumar, Priya Shah, Anand Reddy, Anand Reddy, Vikram Rao, Divya Rao, Karan Iyer, Vikram Nair, Rahul Kumar, Arjun Bose, Arjun Mehta, Anit

# REAL WORLD SCENERIO

In [50]:
import io
import re
import json
import random
import contextlib
import numpy as np
import ollama

EMBED_MODEL = "embeddinggemma:latest"   

In [51]:
# HOSPITAL DATA

random.seed(42)

WARDS = ["ICU", "General", "Pediatric", "Maternity", "Surgical", "Emergency"]
_FIRST = ["Amit", "Sara", "Ravi", "Neha", "Karan", "Divya", "Arjun", "Meera",
          "Rohan", "Anita", "Vikram", "Pooja"]
_LAST = ["Sharma", "Iyer", "Khan", "Reddy", "Nair", "Bose", "Rao", "Mehta"]

PATIENTS = {}
for i in range(300):
    pid = f"P-{2000 + i}"
    PATIENTS[pid] = {
        "id": pid,
        "name": f"{random.choice(_FIRST)} {random.choice(_LAST)}",
        "ward": random.choice(WARDS),
        "age": random.randint(1, 95),
        "admitted_days_ago": random.randint(0, 30),
        "next_med_in_min": random.choice([15, 30, 45, 60, 90, 120, 240, 480]),
        "bill_pending_inr": random.choice([0, 0, 5000, 12000, 45000, 90000]),
        "critical": random.random() < 0.15,
    }

BEDS = {w: {"total": 40, "occupied": random.randint(20, 40)} for w in WARDS}
STAFF = {w: {"doctors": random.randint(2, 6), "nurses": random.randint(5, 15)} for w in WARDS}
DRUGS = {"Paracetamol": 320, "Insulin": 45, "Amoxicillin": 12, "Morphine": 8,
         "Saline": 500, "Adrenaline": 30}

In [70]:
PATIENTS

{'P-2000': {'id': 'P-2000',
  'name': 'Vikram Iyer',
  'ward': 'ICU',
  'age': 95,
  'admitted_days_ago': 8,
  'next_med_in_min': 60,
  'bill_pending_inr': 0,
  'critical': True},
 'P-2001': {'id': 'P-2001',
  'name': 'Sara Iyer',
  'ward': 'Surgical',
  'age': 55,
  'admitted_days_ago': 1,
  'next_med_in_min': 15,
  'bill_pending_inr': 0,
  'critical': False},
 'P-2002': {'id': 'P-2002',
  'name': 'Rohan Sharma',
  'ward': 'Surgical',
  'age': 26,
  'admitted_days_ago': 22,
  'next_med_in_min': 240,
  'bill_pending_inr': 0,
  'critical': False},
 'P-2003': {'id': 'P-2003',
  'name': 'Karan Sharma',
  'ward': 'General',
  'age': 90,
  'admitted_days_ago': 13,
  'next_med_in_min': 120,
  'bill_pending_inr': 5000,
  'critical': False},
 'P-2004': {'id': 'P-2004',
  'name': 'Divya Iyer',
  'ward': 'ICU',
  'age': 49,
  'admitted_days_ago': 3,
  'next_med_in_min': 120,
  'bill_pending_inr': 5000,
  'critical': False},
 'P-2005': {'id': 'P-2005',
  'name': 'Amit Mehta',
  'ward': 'Surgical'

In [56]:
# ══════════════════════════════════════════════════════════════
#  PATIENT TOOLS
# ══════════════════════════════════════════════════════════════
 
def patient_list_ids() -> list[str]:
    """
    Return the unique IDs of every patient currently admitted to the hospital.
 
    Use this as the entry point for any question about the whole patient
    population — counts, totals, averages, or conditional filtering
    ("which patients are critical?", "who has a pending bill over ₹5,000?").
    Pair it with patient_get_bill() or patient_get_next_med_time() in a loop
    inside code for bulk operations, rather than calling patient_get_details()
    for every patient.
 
    Returns:
        list[str]: Patient ID strings for all currently admitted patients.
                   Each ID has the format 'P-XXXX' (e.g. ['P-2050', 'P-1030']).
                   Returns [] if the hospital is currently empty.
    """
    return list(PATIENTS.keys())
 
 
def patient_get_details(patient_id: str) -> dict:
    """
    Retrieve the complete medical and administrative record for a single patient.
 
    Use this when you need the full profile of one specific patient.
    For bulk queries across many patients (counts, filters, totals), iterate
    over patient_list_ids() and call the narrower helpers — patient_get_bill()
    or patient_get_next_med_time() — to avoid loading large records in a loop.
 
    Args:
        patient_id (str):
            The patient's unique identifier.
            Format  : 'P-' followed by exactly four digits.
            Examples: 'P-2050', 'P-0012'.
            Source  : patient_list_ids() or patient_search_by_name().
 
    Returns:
        dict: A full patient record with these fields —
            id               (str)  : Patient's unique ID.
            name             (str)  : Patient's full name.
            ward             (str)  : Current ward. One of: 'ICU', 'General',
                                      'Pediatric', 'Maternity', 'Surgical',
                                      'Emergency'.
            age              (int)  : Patient's age in years.
            admitted_days_ago (int) : Days elapsed since admission.
            next_med_in_min  (int)  : Minutes until next scheduled medication.
                                      0 = due right now.
            bill_pending_inr (int)  : Outstanding bill in Indian Rupees (₹).
                                      0 = account fully settled.
            critical         (bool) : True if the patient requires close
                                      monitoring or is at immediate risk.
 
        On failure —
            error (str): Human-readable message (e.g. unknown ID). Verify
                         the ID with patient_list_ids() and retry.
    """
    return PATIENTS.get(
        patient_id,
        {"error": f"No patient found with ID '{patient_id}'. "
                  f"Call patient_list_ids() to see valid IDs."}
    )
 
 
def patient_search_by_name(name: str) -> list[dict]:
    """
    Search for admitted patients by name using a case-insensitive partial match.
 
    Use this when you have a patient's name but not their ID — for example,
    when a staff member says "find Ramesh Kumar". Returns all patients whose
    name contains the search string anywhere. If you already know the patient
    ID, call patient_get_details() directly.
 
    Args:
        name (str):
            Full or partial name to search for. Case-insensitive.
            Whitespace is stripped before matching.
            Examples: 'ramesh', 'Kumar', 'Priya Sharma'.
 
    Returns:
        list[dict]: Matching patients, each with —
            id   (str) : Patient's unique ID (pass this to other tools).
            name (str) : Patient's full name.
            ward (str) : Ward the patient is currently admitted to.
 
        Returns [] if no patient name matches the query.
    """
    query = name.strip().lower()
    return [
        {"id": pid, "name": p["name"], "ward": p["ward"]}
        for pid, p in PATIENTS.items()
        if query in p["name"].lower()
    ]
 
 
def patient_get_next_med_time(patient_id: str) -> dict:
    """
    Get the minutes remaining until a patient's next scheduled medication dose.
 
    Prefer this over patient_get_details() when looping across all patients
    for medication scheduling — it returns a minimal payload (one field only)
    rather than the full record, which makes bulk loops significantly faster.
 
    Args:
        patient_id (str):
            The patient's unique identifier.
            Format  : 'P-XXXX'  (e.g. 'P-2050').
            Source  : patient_list_ids() or patient_search_by_name().
 
    Returns:
        dict:
            patient_id       (str)       : The patient's ID (echoed back).
            next_med_in_min  (int | None): Minutes from now until the next dose.
                                           0    = medication is due immediately.
                                           None = no medication currently scheduled.
    """
    p = PATIENTS.get(patient_id, {})
    return {
        "patient_id":      patient_id,
        "next_med_in_min": p.get("next_med_in_min"),
    }
 
 
def patient_get_bill(patient_id: str) -> dict:
    """
    Get the outstanding (unpaid) bill balance for a single patient.
 
    Prefer this over patient_get_details() when iterating over many patients
    purely for billing purposes — it returns only the financial field.
    Use billing_get_total_outstanding() if you only need the hospital-wide
    sum and do not need to identify individual patients.
 
    Args:
        patient_id (str):
            The patient's unique identifier.
            Format  : 'P-XXXX'  (e.g. 'P-2050').
            Source  : patient_list_ids() or patient_search_by_name().
 
    Returns:
        dict:
            patient_id       (str) : The patient's ID (echoed back).
            bill_pending_inr (int) : Unpaid bill in Indian Rupees (₹).
                                     0 = account is fully settled.
    """
    p = PATIENTS.get(patient_id, {})
    return {
        "patient_id":      patient_id,
        "bill_pending_inr": p.get("bill_pending_inr"),
    }
 
 
# ══════════════════════════════════════════════════════════════
#  WARD / BED TOOLS
# ══════════════════════════════════════════════════════════════
 
def ward_list_all() -> list[str]:
    """
    Return the names of every ward in the hospital.
 
    Use this to discover valid ward names before calling any ward-specific
    tool, or to iterate over all wards (e.g. to check occupancy or staffing
    levels across the whole hospital).
 
    Returns:
        list[str]: All ward name strings.
                   Current wards: ['ICU', 'General', 'Pediatric',
                                   'Maternity', 'Surgical', 'Emergency'].
    """
    return WARDS
 
 
def ward_get_status(ward: str) -> dict:
    """
    Get the complete operational status of a ward — bed occupancy AND
    current on-duty staff — in a single call.
 
    Use this before admitting or transferring a patient to confirm there
    is both a free bed and adequate staffing. This call replaces the
    previous separate ward_bed_availability() and ward_staff_count() calls.
 
    Args:
        ward (str):
            Name of the ward to query. Case-sensitive.
            Valid values:
                'ICU'       — Intensive Care Unit
                'General'   — General ward
                'Pediatric' — Pediatric ward
                'Maternity' — Maternity ward
                'Surgical'  — Surgical ward
                'Emergency' — Emergency ward
            Call ward_list_all() to retrieve valid names programmatically.
 
    Returns:
        dict:
            ward     (str) : Ward name (echoed back).
            total    (int) : Total beds in the ward.
            occupied (int) : Beds currently in use.
            free     (int) : Available beds (total − occupied). Admit only
                             if free > 0.
            doctors  (int) : Doctors currently on duty.
            nurses   (int) : Nurses currently on duty.
 
        On failure —
            error (str): 'unknown ward' if the ward name is not recognised.
                         Call ward_list_all() to verify valid names.
    """
    bed_data   = BEDS.get(ward, {})
    staff_data = STAFF.get(ward, {})
 
    if not bed_data or not staff_data:
        return {"ward": ward, "error": "unknown ward — call ward_list_all() for valid names"}
 
    total    = bed_data["total"]
    occupied = bed_data["occupied"]
    return {
        "ward":     ward,
        "total":    total,
        "occupied": occupied,
        "free":     total - occupied,
        "doctors":  staff_data["doctors"],
        "nurses":   staff_data["nurses"],
    }
 
 
# ══════════════════════════════════════════════════════════════
#  PHARMACY TOOLS
# ══════════════════════════════════════════════════════════════
 
def pharmacy_list_drugs() -> list[str]:
    """
    Return the names of all drugs currently tracked by the pharmacy.
 
    Use this to discover valid drug names before calling pharmacy_check_stock()
    or pharmacy_reorder(). Both those functions require an exact, case-sensitive
    drug name; this call lets you look it up first.
 
    Returns:
        list[str]: Drug names for every item in the pharmacy system.
                   Example: ['Insulin', 'Paracetamol', 'Metformin', 'Amoxicillin'].
    """
    return list(DRUGS.keys())
 
 
def pharmacy_check_stock(drug_name: str) -> dict:
    """
    Check how many units of a specific drug are currently available in the pharmacy.
 
    Always call this before administering a drug or placing a reorder, to
    confirm current availability. If the drug name is unknown, call
    pharmacy_list_drugs() first to see exact names in the system.
 
    Args:
        drug_name (str):
            The drug's name as it appears in the pharmacy system.
            Case-sensitive. Examples: 'Insulin', 'Paracetamol', 'Metformin'.
            Call pharmacy_list_drugs() to see all valid names.
 
    Returns:
        dict:
            drug           (str) : The drug name (echoed back).
            units_in_stock (int) : Units currently available.
                                   0 = completely out of stock; call
                                   pharmacy_reorder() immediately.
    """
    return {"drug": drug_name, "units_in_stock": DRUGS.get(drug_name, 0)}
 
 
def pharmacy_reorder(drug_name: str, units: int) -> dict:
    """
    Place a restock order for a drug with the pharmacy supplier.
 
    Always call pharmacy_check_stock() first to confirm the drug is actually
    low or out of stock before ordering, to prevent over-ordering.
 
    Args:
        drug_name (str):
            The drug's name as it appears in the pharmacy system.
            Case-sensitive. Examples: 'Insulin', 'Paracetamol'.
            Call pharmacy_list_drugs() to see all valid names.
        units (int):
            Number of units to reorder.
            Must be a positive integer (minimum: 1).
            Examples: 100, 500.
 
    Returns:
        dict:
            ok              (bool) : True = order placed successfully.
            drug            (str)  : Drug name (echoed back).
            reordered_units (int)  : Quantity ordered (echoed back).
    """
    return {"ok": True, "drug": drug_name, "reordered_units": units}
 
 
# ══════════════════════════════════════════════════════════════
#  BILLING TOOLS
# ══════════════════════════════════════════════════════════════
 
def billing_get_total_outstanding() -> dict:
    """
    Get the hospital-wide total of all unpaid patient bills as a single figure.
 
    Use this only when you need the aggregate sum, without identifying which
    specific patients owe money. To find WHICH patients have outstanding
    balances, or to filter by amount, iterate over patient_list_ids() and
    call patient_get_bill() in code.
 
    Returns:
        dict:
            total_outstanding_inr (int): Sum of all pending bills across every
                                         admitted patient, in Indian Rupees (₹).
                                         0 = all patient accounts are fully settled.
    """
    return {"total_outstanding_inr": sum(p["bill_pending_inr"] for p in PATIENTS.values())}
 
 
def billing_mark_paid(patient_id: str) -> dict:
    """
    Mark a patient's entire outstanding bill as fully paid and clear their balance.
 
    Use this after full payment is confirmed. This clears the entire pending
    amount — it does not support partial payments. Before calling, verify the
    current balance with patient_get_bill() so you can confirm the correct
    amount with the payer.
 
    Args:
        patient_id (str):
            The patient's unique identifier.
            Format  : 'P-XXXX'  (e.g. 'P-2050').
            Source  : patient_list_ids() or patient_search_by_name().
 
    Returns:
        dict:
            ok         (bool) : True = bill successfully cleared.
            patient_id (str)  : Patient's ID (echoed back).
            status     (str)  : 'paid' — balance is now ₹0.
    """
    return {"ok": True, "patient_id": patient_id, "status": "paid"}
 
 
# ══════════════════════════════════════════════════════════════
#  APPOINTMENT TOOLS
# ══════════════════════════════════════════════════════════════
 
def appointment_list_for_patient(patient_id: str) -> list[dict]:
    """
    List all upcoming outpatient appointments already booked for a patient.
 
    Always call this BEFORE booking a new appointment to check for existing
    bookings and avoid duplicate or conflicting slots.
 
    Args:
        patient_id (str):
            The patient's unique identifier.
            Format  : 'P-XXXX'  (e.g. 'P-2050').
            Source  : patient_list_ids() or patient_search_by_name().
 
    Returns:
        list[dict]: Upcoming appointments, each with —
            appointment_id (str) : Unique appointment ID (format: 'A-XXX').
                                   Pass this to appointment_cancel() if needed.
            department     (str) : Clinic or specialist department name.
            day            (str) : Scheduled day (e.g. 'Monday', 'Wednesday').
 
        Returns [] if the patient has no upcoming appointments.
    """
    return [{"appointment_id": "A-501", "department": "Cardiology", "day": "Monday"}]
 
 
def appointment_book(patient_id: str, department: str, day: str) -> dict:
    """
    Book a new outpatient appointment for a patient in a specialist department.
 
    Always call appointment_list_for_patient() first to check for conflicts
    before creating a new booking.
 
    Args:
        patient_id (str):
            The patient's unique identifier.
            Format  : 'P-XXXX'  (e.g. 'P-2050').
            Source  : patient_list_ids() or patient_search_by_name().
        department (str):
            The clinic or specialist department to book into.
            Examples: 'Cardiology', 'Neurology', 'Orthopedics',
                      'Ophthalmology', 'ENT', 'Dermatology', 'Oncology'.
        day (str):
            Day of the week for the appointment.
            Valid values : 'Monday', 'Tuesday', 'Wednesday',
                           'Thursday', 'Friday', 'Saturday'.
            Note         : Appointments are NOT available on Sundays.
 
    Returns:
        dict:
            ok             (bool) : True = booking confirmed.
            appointment_id (str)  : Newly generated unique ID (format: 'A-XXX').
                                    Save this to cancel later.
            patient_id     (str)  : Patient's ID (echoed back).
            department     (str)  : Department booked (echoed back).
            day            (str)  : Day booked (echoed back).
    """
    return {
        "ok":             True,
        "appointment_id": f"A-{random.randint(100, 999)}",
        "patient_id":     patient_id,
        "department":     department,
        "day":            day,
    }
 
 
def appointment_cancel(appointment_id: str) -> dict:
    """
    Cancel an existing outpatient appointment by its appointment ID.
 
    Use when a patient requests cancellation or needs to reschedule. To
    reschedule, call this first then call appointment_book() with the new
    details. Retrieve the appointment ID from appointment_list_for_patient().
 
    Args:
        appointment_id (str):
            The unique appointment ID to cancel.
            Format : 'A-XXX'  (e.g. 'A-501').
            Source : appointment_list_for_patient().
 
    Returns:
        dict:
            ok             (bool) : True = cancellation processed.
            appointment_id (str)  : Cancelled appointment ID (echoed back).
            status         (str)  : 'cancelled' — the slot is now free.
    """
    return {"ok": True, "appointment_id": appointment_id, "status": "cancelled"}
 
 
# ══════════════════════════════════════════════════════════════
#  LAB TOOLS
# ══════════════════════════════════════════════════════════════
 
def lab_list_tests_for_patient(patient_id: str) -> list[dict]:
    """
    List all lab tests ordered for a patient and their current processing status.
 
    Call this BEFORE ordering a new test to avoid duplicate orders, and to
    know which test names to pass to lab_get_result() once processing finishes.
 
    Args:
        patient_id (str):
            The patient's unique identifier.
            Format  : 'P-XXXX'  (e.g. 'P-2050').
            Source  : patient_list_ids() or patient_search_by_name().
 
    Returns:
        list[dict]: Ordered tests, each with —
            order_id (str) : Lab order ID (format: 'L-XXX').
            test     (str) : Test name (e.g. 'CBC', 'X-Ray', 'LFT').
            status   (str) : Current state. One of:
                               'pending'     — Not yet started.
                               'in_progress' — Currently being processed.
                               'completed'   — Result is ready; call
                                               lab_get_result() to retrieve it.
 
        Returns [] if no tests have been ordered for this patient.
    """
    return []   # Replace with real data source
 
 
def lab_order_test(patient_id: str, test_name: str) -> dict:
    """
    Order a laboratory or diagnostic test for a patient.
 
    Call lab_list_tests_for_patient() first to confirm the test has not
    already been ordered. Once placed, monitor status via that same function
    and retrieve the result with lab_get_result() when status = 'completed'.
 
    Args:
        patient_id (str):
            The patient's unique identifier.
            Format  : 'P-XXXX'  (e.g. 'P-2050').
            Source  : patient_list_ids() or patient_search_by_name().
        test_name (str):
            Name of the test to order.
            Blood tests : 'CBC', 'LFT', 'RFT', 'Blood Sugar',
                          'Lipid Profile', 'Thyroid Profile'
            Imaging     : 'X-Ray', 'MRI', 'CT Scan', 'Ultrasound'
            Other       : 'ECG', 'Urine Culture', 'Biopsy'
 
    Returns:
        dict:
            ok         (bool) : True = order placed successfully.
            order_id   (str)  : Newly generated lab order ID (format: 'L-XXX').
            patient_id (str)  : Patient's ID (echoed back).
            test       (str)  : Test name (echoed back).
    """
    return {
        "ok":         True,
        "order_id":   f"L-{random.randint(100, 999)}",
        "patient_id": patient_id,
        "test":       test_name,
    }
 
 
def lab_get_result(patient_id: str, test_name: str) -> dict:
    """
    Retrieve the result of a completed laboratory test for a patient.
 
    Only call this after lab_list_tests_for_patient() shows status = 'completed'
    for the test. Calling it earlier may return incomplete or placeholder data.
 
    Args:
        patient_id (str):
            The patient's unique identifier.
            Format  : 'P-XXXX'  (e.g. 'P-2050').
            Source  : patient_list_ids() or patient_search_by_name().
        test_name (str):
            Exact name of the test whose result you want.
            Must match the name used when lab_order_test() was called.
            Examples: 'CBC', 'X-Ray', 'LFT'.
 
    Returns:
        dict:
            patient_id (str) : Patient's ID (echoed back).
            test       (str) : Test name (echoed back).
            result     (str) : Outcome of the test.
                               Examples: 'normal',
                                         'abnormal — see attached report',
                                         'borderline',
                                         'critical — immediate review required'.
    """
    return {"patient_id": patient_id, "test": test_name, "result": "normal"}
 
 
# ══════════════════════════════════════════════════════════════
#  EMERGENCY / ADMIN TOOLS
# ══════════════════════════════════════════════════════════════
 
def emergency_alert_team(ward: str) -> dict:
    """
    ⚠️  HIGH-IMPACT — Dispatch an urgent alert to the rapid-response emergency
    team for a specific ward.
 
    Use ONLY for genuine life-threatening emergencies requiring immediate
    physical intervention. Do NOT use for routine escalations, medication
    delays, or administrative issues. Before triggering:
      • Confirm the patient is flagged critical via patient_get_details()
        (critical = True), OR
      • Confirm the ward is dangerously over capacity via ward_get_status().
 
    Args:
        ward (str):
            Ward requiring emergency response. Case-sensitive.
            Valid values:
                'ICU', 'General', 'Pediatric',
                'Maternity', 'Surgical', 'Emergency'.
            Call ward_list_all() to retrieve valid names programmatically.
 
    Returns:
        dict:
            ok           (bool) : True = alert successfully dispatched.
            alerted_ward (str)  : Ward the team was alerted to (echoed back).
    """
    return {"ok": True, "alerted_ward": ward}
 
 
def admin_get_hospital_summary() -> dict:
    """
    Get a high-level snapshot of the hospital — total patient count and
    number of wards — as a single fast call.
 
    Use this for a quick top-level overview only. For detailed per-patient
    or per-ward data, use the specific patient or ward tools.
 
    Returns:
        dict:
            total_patients (int) : Total patients currently admitted.
            wards          (int) : Total number of wards in the hospital.
    """
    return {"total_patients": len(PATIENTS), "wards": len(WARDS)}
 
 
def admin_transfer_patient(patient_id: str, to_ward: str) -> dict:
    """
    Transfer an admitted patient from their current ward to a different one.
 
    Before calling this:
      1. Confirm the destination ward has a free bed and adequate staff
         via ward_get_status() (free > 0).
      2. If the patient is critical (patient_get_details() → critical: True),
         confirm the destination ward has sufficient staff; if not, call
         emergency_alert_team() as a follow-up after the transfer.
 
    Args:
        patient_id (str):
            The patient's unique identifier.
            Format  : 'P-XXXX'  (e.g. 'P-2050').
            Source  : patient_list_ids() or patient_search_by_name().
        to_ward (str):
            Destination ward name. Case-sensitive.
            Valid values:
                'ICU', 'General', 'Pediatric',
                'Maternity', 'Surgical', 'Emergency'.
            Call ward_list_all() to retrieve valid names programmatically.
 
    Returns:
        dict:
            ok         (bool) : True = transfer completed.
            patient_id (str)  : Patient's ID (echoed back).
            from_ward  (str)  : Ward the patient was moved FROM.
            to_ward    (str)  : Ward the patient was moved TO.
    """
    from_ward = PATIENTS.get(patient_id, {}).get("ward", "Unknown")
    return {
        "ok":         True,
        "patient_id": patient_id,
        "from_ward":  from_ward,
        "to_ward":    to_ward,
    }
 
 
def admin_discharge_patient(patient_id: str) -> dict:
    """
    ⚠️  IRREVERSIBLE — Discharge a patient from the hospital, freeing their
    bed for a new admission.
 
    Once called, the patient's bed is released and their admission record is
    closed. This cannot be undone. Before discharging, verify:
      1. Outstanding bill is ₹0  → patient_get_bill() → bill_pending_inr == 0
         (or settle it first with billing_mark_paid()).
      2. No critical flag is set → patient_get_details() → critical == False.
      3. All lab tests are resolved → lab_list_tests_for_patient() shows
         no 'pending' or 'in_progress' tests.
 
    Args:
        patient_id (str):
            The patient's unique identifier.
            Format  : 'P-XXXX'  (e.g. 'P-2050').
            Source  : patient_list_ids() or patient_search_by_name().
 
    Returns:
        dict:
            ok         (bool) : True = discharge processed successfully.
            patient_id (str)  : Patient's ID (echoed back).
            status     (str)  : 'discharged' — bed is now free.
    """
    return {"ok": True, "patient_id": patient_id, "status": "discharged"}
 
 
# ══════════════════════════════════════════════════════════════
#  REGISTRY
# ══════════════════════════════════════════════════════════════
 
REAL_TOOLS = [
    # Patient
    patient_list_ids, patient_get_details, patient_search_by_name,
    patient_get_next_med_time, patient_get_bill,
    # Ward
    ward_list_all, ward_get_status,
    # Pharmacy
    pharmacy_list_drugs, pharmacy_check_stock, pharmacy_reorder,
    # Billing
    billing_get_total_outstanding, billing_mark_paid,
    # Appointments
    appointment_list_for_patient, appointment_book, appointment_cancel,
    # Lab
    lab_list_tests_for_patient, lab_order_test, lab_get_result,
    # Emergency / Admin
    emergency_alert_team, admin_get_hospital_summary,
    admin_transfer_patient, admin_discharge_patient,
]
 
REGISTRY = {f.__name__: f for f in REAL_TOOLS}
print(f"{len(REAL_TOOLS)} tools loaded")

22 tools loaded


In [57]:
REGISTRY

{'patient_list_ids': <function __main__.patient_list_ids() -> list[str]>,
 'patient_get_details': <function __main__.patient_get_details(patient_id: str) -> dict>,
 'patient_search_by_name': <function __main__.patient_search_by_name(name: str) -> list[dict]>,
 'patient_get_next_med_time': <function __main__.patient_get_next_med_time(patient_id: str) -> dict>,
 'patient_get_bill': <function __main__.patient_get_bill(patient_id: str) -> dict>,
 'ward_list_all': <function __main__.ward_list_all() -> list[str]>,
 'ward_get_status': <function __main__.ward_get_status(ward: str) -> dict>,
 'pharmacy_list_drugs': <function __main__.pharmacy_list_drugs() -> list[str]>,
 'pharmacy_check_stock': <function __main__.pharmacy_check_stock(drug_name: str) -> dict>,
 'pharmacy_reorder': <function __main__.pharmacy_reorder(drug_name: str, units: int) -> dict>,
 'billing_get_total_outstanding': <function __main__.billing_get_total_outstanding() -> dict>,
 'billing_mark_paid': <function __main__.billing_

In [58]:
## HELPER FUNCTIONS FOR HYBRID SEARC [DEFERRED LOADING]
def get_embedding(text, task_type="search_document"):
    response = ollama.embed(model=EMBED_MODEL, input=f"{task_type}: {text}")
    return response["embeddings"][0]

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def clean_docstring(doc):
    if not doc:
        return ""
    return re.split(r"\n\s*(Args:|Returns:)", doc)[0].strip()

In [59]:
# precompute embeddings ONCE
TOOL_EMBEDDINGS = {}
for f in REAL_TOOLS:
    text = f.__name__ + " " + clean_docstring(f.__doc__)
    TOOL_EMBEDDINGS[f.__name__] = get_embedding(text, task_type="search_document")
print(f"Precomputed embeddings for {len(TOOL_EMBEDDINGS)} tools")

Precomputed embeddings for 22 tools


In [62]:
def search_tools(query, max_results=4):
    max_results = int(max_results)          # <-- ADD THIS LINE
    query_emb = get_embedding(query, task_type="search_query")
    query_words = set(query.lower().split())
    scored = []
    for f in REAL_TOOLS:
        name = f.__name__
        text = (name + " " + clean_docstring(f.__doc__)).lower()
        tool_words = set(text.split())
        keyword_score = len(query_words & tool_words) / max(len(query_words), 1)
        embedding_score = cosine_similarity(query_emb, TOOL_EMBEDDINGS[name])
        combined = (0.6 * keyword_score) + (0.4 * embedding_score)
        scored.append((combined, name))
    scored.sort(reverse=True)
    return [name for score, name in scored[:max_results]]

In [63]:
SEARCH_TOOL = {
    "type": "function",
    "function": {
        "name": "search_tools",
        "description": (
            "Search for hospital tools not yet loaded into this conversation, "
            "by meaning as well as keywords. Use this the moment you realize you "
            "need a capability you don't currently have (e.g. checking beds, "
            "bills, staffing, pharmacy).\n\n"
            "Returns matching tool names ranked by relevance. Once a tool is "
            "returned here it becomes directly callable on your next turn - you "
            "do not need to search for it again. If the top result looks wrong, "
            "rephrase the query rather than giving up."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": (
                    "Describe the capability you need as a natural phrase - full "
                    "context helps more than bare keywords.")},
                "max_results": {"type": "integer",
                                "description": "Max tools to return. Defaults to 4."},
            },
            "required": ["query"],
        },
    },
}


In [64]:
def run_code(python_code: str):
    """Execute Python code to answer questions requiring computation across
    many patients or wards - counts, totals, averages, or filtering.

    All hospital tool functions are available directly inside your code,
    including patient_list_ids(), patient_get_next_med_time(id),
    patient_get_bill(id), and patient_get_details(id). Write a loop over the
    patients, compute the result, and print ONLY the final answer - the raw
    per-patient data stays inside the code and never enters the conversation.

    Args:
        python_code: The Python code to run. The hospital tool functions are
            already in scope; do not import or redefine them.

    Returns:
        A dict with 'result' (whatever the code printed) or 'error' (the
        exception message, so the code can be corrected and retried).
    """
    workspace = dict(REGISTRY)
    buffer = io.StringIO()
    try:
        with contextlib.redirect_stdout(buffer):
            exec(python_code, workspace)
        return {"result": buffer.getvalue().strip() or "(code ran but printed nothing)"}
    except Exception as e:
        return {"error": f"{type(e).__name__}: {e}"}

In [66]:
RUN_CODE_TOOL = {
    "type": "function",
    "function": {
        "name": "run_code",
        "description": clean_docstring(run_code.__doc__),
        "parameters": {
            "type": "object",
            "properties": {"python_code": {"type": "string", "description":
                "Python code to run. Hospital tool functions are already in scope."}},
            "required": ["python_code"],
        },
    },
}


In [67]:
SYSTEM = """
You are an advanced Hospital Operations AI Assistant designed to help hospital administrators and clinical staff manage resources, 
patient data, and billing. You are professional, highly accurate, and strictly rely on the provided operational databases.

### 1. DATA DICTIONARY (Schema for programmatic calling)
When writing code or answering data-related queries, you must use these EXACT dictionary keys and table structures. Do not guess or invent column names.

**PATIENTS Table** (Accessed via `patient_get_details(pid)` or iterating `patient_list_ids()`)
- `id` (str): Unique patient ID (e.g., 'P-2000').
- `name` (str): Full name of the patient.
- `ward` (str): The ward where the patient is admitted (Must be one of: 'ICU', 'General', 'Pediatric', 'Maternity', 'Surgical', 'Emergency').
- `age` (int): Patient's age in years.
- `admitted_days_ago` (int): Days since admission (0-30).
- `next_med_in_min` (int): Minutes until the next scheduled medication.
- `bill_pending_inr` (int): Outstanding bill amount in INR (Indian Rupees).
- `critical` (bool): True if the patient is in critical condition, False otherwise.

**BEDS / WARDS Table** (Accessed via `ward_bed_availability(ward)`)
- `ward` (str): Name of the ward.
- `total` (int): Total number of beds in the ward.
- `occupied` (int): Number of currently occupied beds.

**STAFF Table** (Accessed via `ward_staff_count(ward)`)
- `ward` (str): Name of the ward.
- `doctors` (int): Number of doctors currently on duty.
- `nurses` (int): Number of nurses currently on duty.

**DRUGS / PHARMACY Table** (Accessed via `pharmacy_check_stock(drug_name)`)
- `drug` (str): Name of the medication (e.g., 'Paracetamol', 'Insulin').
- `units_in_stock` (int): Number of units currently available.


### 2. TOOL USAGE STRATEGY
You begin with only two meta-tools: `search_tools` and `run_code`.
- **For SINGLE targeted queries** (e.g., "Check stock for Insulin", "Discharge patient P-2050"):
  Use `search_tools` to find the exact administrative capability you need, then call that newly loaded tool.
- **For AGGREGATE/BULK queries** (e.g., "Average age of ICU patients", "List all patients with bills > 50000"):
  Use `run_code` to execute a Python script. Do not fetch records one by one. Iterate over `patient_list_ids()` and use the exact keys from the Data Dictionary above.


### 3. CODING GUIDELINES (When using run_code)
- **Efficiency**: Use efficient loops. All hospital tool functions are already in scope.
- **Strict Schema**: Only reference the keys explicitly defined in the Data Dictionary. 
- **Output Validation**: You must `print()` the final result inside your code. Do not just return it, or the output buffer will be empty. The raw per-patient data must stay inside the script.


### 4. CONSTRAINTS & TONE
- Never hallucinate patient data, staff counts, or medical records. If data is not found, state it clearly.
- Provide concise, direct answers. Medical staff are busy; avoid unnecessary conversational filler.
- You are an operational assistant, not a doctor. Never provide medical diagnoses or treatment advice.
"""

In [68]:
def run_agent(question, max_turns=8):
    loaded = [SEARCH_TOOL, RUN_CODE_TOOL]
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": question}]

    print(f"\n{'='*70}\nQ: {question}\n{'='*70}")

    for turn in range(1, max_turns + 1):
        response = ollama.chat(model=MODEL, messages=messages, tools=loaded)
        size = response.prompt_eval_count or 0
        print(f"turn {turn}: {size:>5} tokens | tools loaded: {len(loaded)}")
        messages.append(response.message)

        if not response.message.tool_calls:
            print(f"ANSWER: {response.message.content}\n")
            break

        for call in response.message.tool_calls:
            name, args = call.function.name, call.function.arguments

            if name == "search_tools":
                found = search_tools(**args)
                print(f"   [search] {args.get('query')!r} -> {found}")
                for n in found:
                    if all(t["function"]["name"] != n for t in loaded):
                        loaded.append({"type": "function", "function": {
                            "name": n,
                            "description": clean_docstring(REGISTRY[n].__doc__),
                            "parameters": {"type": "object", "properties": {},
                                           "required": []}}})
                messages.append({"role": "tool", "tool_name": "search_tools",
                                 "content": json.dumps({"loaded": found})})

            elif name == "run_code":
                code = args["python_code"]
                print(f"   [run_code] {len(code)} chars")
                result = run_code(code)
                messages.append({"role": "tool", "tool_name": "run_code",
                                 "content": json.dumps(result)})

            else:
                print(f"   [call] {name}({args})")
                result = REGISTRY[name](**args)
                messages.append({"role": "tool", "tool_name": name,
                                 "content": json.dumps(result)})

In [71]:
#NORMAL RUNS
run_agent("What is the pending bill for patient  Karan Sharma")  #Karan Sharma  Vikram Iyer?


Q: What is the pending bill for patient  Karan Sharma
turn 1:  1219 tokens | tools loaded: 2
   [run_code] 287 chars
turn 2:  1353 tokens | tools loaded: 2
ANSWER: The pending bill for patient Karan Sharma (ID: P-2003) is 5,000 INR.



In [72]:
run_agent("Is Insulin in stock in the pharmacy?")


Q: Is Insulin in stock in the pharmacy?
turn 1:  1216 tokens | tools loaded: 2
   [search] 'check pharmacy stock for a specific drug' -> ['pharmacy_reorder', 'pharmacy_check_stock', 'admin_get_hospital_summary', 'appointment_list_for_patient']
turn 2:  1590 tokens | tools loaded: 6
   [call] pharmacy_check_stock({'drug_name': 'Insulin'})
turn 3:  1644 tokens | tools loaded: 6
ANSWER: Yes, Insulin is in stock with 45 units available.



In [17]:
# COMPUTATION QUERIES

In [73]:
run_agent("Which patients have a pending bill over 40000 rupees? List their names and their count")


Q: Which patients have a pending bill over 40000 rupees? List their names and their count
turn 1:  1229 tokens | tools loaded: 2
   [run_code] 308 chars
turn 2:  1693 tokens | tools loaded: 2
ANSWER: The following 99 patients have a pending bill over 40,000 rupees:

**Count:** 99
**Patients:** Divya Khan, Arjun Mehta, Sara Sharma, Sara Rao, Amit Iyer, Karan Rao, Anita Rao, Neha Nair, Vikram Rao, Rohan Sharma, Vikram Reddy, Pooja Bose, Ravi Mehta, Karan Iyer, Divya Khan, Rohan Sharma, Meera Bose, Divya Nair, Karan Sharma, Amit Reddy, Karan Khan, Arjun Bose, Meera Iyer, Neha Khan, Pooja Rao, Meera Nair, Rohan Iyer, Arjun Mehta, Meera Khan, Sara Sharma, Karan Iyer, Divya Mehta, Arjun Mehta, Meera Sharma, Karan Reddy, Karan Sharma, Sara Rao, Meera Nair, Rohan Iyer, Karan Sharma, Anita Nair, Anita Nair, Anita Reddy, Rohan Rao, Arjun Reddy, Anita Rao, Vikram Khan, Vikram Rao, Karan Reddy, Ravi Reddy, Karan Khan, Sara Mehta, Arjun Bose, Meera Sharma, Vikram Rao, Neha Iyer, Amit Khan, Ravi Na

In [74]:
run_agent("What is the average age of all patients across the hospital?")


Q: What is the average age of all patients across the hospital?
turn 1:  1220 tokens | tools loaded: 2
   [run_code] 265 chars
turn 2:  1349 tokens | tools loaded: 2
ANSWER: The average age of all patients across the hospital is 47.87 years.



In [75]:
run_agent("How many patients have age between 12,18")


Q: How many patients have age between 12,18
turn 1:  1220 tokens | tools loaded: 2
   [run_code] 146 chars
turn 2:  1304 tokens | tools loaded: 2
ANSWER: There are 30 patients between the ages of 12 and 18.



In [76]:
run_agent("Give me number of patients as per each ward")


Q: Give me number of patients as per each ward
turn 1:  1217 tokens | tools loaded: 2
   [run_code] 262 chars
turn 2:  1367 tokens | tools loaded: 2
ANSWER: The number of patients per ward is as follows:

- **Surgical**: 62
- **General**: 52
- **ICU**: 50
- **Emergency**: 49
- **Maternity**: 45
- **Pediatric**: 42



In [77]:
run_agent("Give me count of patients who  need medicine in next 60 minutes.")


Q: Give me count of patients who  need medicine in next 60 minutes.
turn 1:  1224 tokens | tools loaded: 2
   [run_code] 200 chars
turn 2:  1329 tokens | tools loaded: 2
ANSWER: 151 patients need medication in the next 60 minutes.



In [78]:
run_agent("How many patients are currently in critical condition.")


Q: How many patients are currently in critical condition.
turn 1:  1217 tokens | tools loaded: 2
   [run_code] 213 chars
turn 2:  1316 tokens | tools loaded: 2
ANSWER: There are currently 43 patients in critical condition.



In [79]:
run_agent("How many males and females patients are there in each kind of ward.")


Q: How many males and females patients are there in each kind of ward.
turn 1:  1222 tokens | tools loaded: 2
   [run_code] 630 chars
turn 2:  1582 tokens | tools loaded: 2
ANSWER: Based on the available patient data, gender information is not recorded in the database. Consequently, I can only provide the total patient counts per ward:

- **Surgical**: 62 patients
- **General**: 52 patients
- **ICU**: 50 patients
- **Emergency**: 49 patients
- **Maternity**: 45 patients
- **Pediatric**: 42 patients

